In [23]:
import json
import requests
from openai import OpenAI

client = OpenAI()


def get_popular_movies():
    url = "https://nomad-movies.nomadcoders.workers.dev/movies"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_details(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


def get_movie_credits(id):
    url = f"https://nomad-movies.nomadcoders.workers.dev/movies/{id}/credits"
    resp = requests.get(url, timeout=10)
    resp.raise_for_status()
    return resp.json()


TOOLS = [
    {
        "type": "function",
        "name": "get_popular_movies",
        "description": "Get a list of currently popular movies.",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_details",
        "description": "Get detailed information about a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
    {
        "type": "function",
        "name": "get_movie_credits",
        "description": "Get cast and crew credits for a movie by id.",
        "parameters": {
            "type": "object",
            "properties": {
                "id": {
                    "type": "integer",
                    "description": "The movie id"
                }
            },
            "required": ["id"],
            "additionalProperties": False,
        },
    },
]


def call_tool(tool_name, args):
    if tool_name == "get_popular_movies":
        return get_popular_movies()
    elif tool_name == "get_movie_details":
        return get_movie_details(args["id"])
    elif tool_name == "get_movie_credits":
        return get_movie_credits(args["id"])
    else:
        raise ValueError(f"Unknown tool: {tool_name}")


def run_agent(user_input):
    response = client.responses.create(
        model="gpt-4o-mini",
        input=user_input,
        instructions=("You are a helpful movie assistant. "
        "Use the provided tools whenever the user asks for real movie data. " 
        "If the user asks for popular movies, use get_popular_movies. " 
        "If the user asks for details, use get_movie_details. " 
        "If the user asks for cast or crew, use get_movie_credits. " "Answer clearly in Korean."
        ),
        tools=TOOLS,
    )

    function_calls = [
        item for item in response.output
        if item.type == "function_call"
    ]

    if not function_calls:
        return response.output_text

    call = function_calls[0]

    tool_name = call.name
    args = json.loads(call.arguments)

    print("🔧 Function Call:", tool_name, args)

    tool_result = call_tool(tool_name, args)

    final_response = client.responses.create(
        model="gpt-4o-mini",
        previous_response_id=response.id,
        input=[{
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(tool_result, ensure_ascii=False),
        }],
    )

    return final_response.output_text


In [25]:

result = run_agent("지금 인기 있는 영화가 뭐야?")
print(result)


🔧 Function Call: get_popular_movies {}
현재 인기 있는 영화 몇 편을 소개할게요:

1. **Your Heart Will Be Broken**
   - **개요**: 고등학생 폴리나는 학교에서 괴롭힘을 당하는 중, 주범인 바르스와 연인 행세를 하며 진짜 감정을 키워가지만, 가족과 학급 친구들이 둘 사이를 갈라놓으려 합니다.
   - **개봉일**: 2026년 3월 26일
   - **평점**: 6.5
   - ![포스터](https://image.tmdb.org/t/p/w780/iGpMm603GUKH2SiXB2S5m4sZ17t.jpg)

2. **Avatar: Fire and Ash**
   - **개요**: Jake Sully와 Neytiri는 새로운 적, 'Ash People'의 위협에 맞서 싸워야 합니다.
   - **개봉일**: 2025년 12월 17일
   - **평점**: 7.4
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - **개요**: 훌륭한 로봇 동물과 의사소통할 수 있는 기술을 통해 동물의 세계의 신비를 탐험하는 이야기입니다.
   - **개봉일**: 2026년 3월 4일
   - **평점**: 7.6
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **Crime 101**
   - **개요**: 로스앤젤레스에서의 고난이도 범죄를 다루고 있으며, 스릴 넘치는 스토리입니다.
   - **개봉일**: 2026년 2월 11일
   - **평점**: 7.1
   - ![포스터](https://image.tmdb.org/t/p/w780/tVvpFIoteRHNnoZMhdnwIVwJpCA.jpg)

이 영화들이 현재 주목받고 있습니다! 더 알고 싶은 내용이 있다면 말씀해 주세요.


In [27]:

result = run_agent("movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘")
print(result)


🔧 Function Call: get_movie_credits {'id': 550}
영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 이 영화의 주요 출연진은 다음과 같습니다:

1. **Edward Norton** - Narrator  
   ![Edward Norton](https://image.tmdb.org/t/p/w185/8nytsqL59SFJTVYVrN72k6qkGgJ.jpg)

2. **Brad Pitt** - Tyler Durden  
   ![Brad Pitt](https://image.tmdb.org/t/p/w185/r9DzKQLNbh5QfXlrFGHoVNKER7X.jpg)

3. **Helena Bonham Carter** - Marla Singer  
   ![Helena Bonham Carter](https://image.tmdb.org/t/p/w185/hJMbNSPJ2PCahsP3rNEU39C8GWU.jpg)

4. **Meat Loaf** - Robert Paulson  
   ![Meat Loaf](https://image.tmdb.org/t/p/w185/7gKLR1u46OB8WJ6m06LemNBCMx6.jpg)

5. **Jared Leto** - Angel Face  
   ![Jared Leto](https://image.tmdb.org/t/p/w185/ca3x0OfIKbJppZh8S1Alx3GfUZO.jpg)

이 외에도 많은 배우들이 출연하였습니다. 더 알고 싶으신 배우나 정보가 있으시면 말씀해 주세요!


In [26]:

result = run_agent("movie ID 550에 해당하는 영화가 무엇인지 알려줘")
print(result)


🔧 Function Call: get_movie_details {'id': 550}
영화 ID 550에 해당하는 영화는 **"Fight Club"**입니다. 

### 기본 정보:
- **개봉일**: 1999년 10월 15일
- **장르**: 드라마, 스릴러
- **러닝타임**: 139분
- **예산**: $63,000,000
- **수익**: $100,853,753
- **평점**: 8.4 (투표 수: 31,729)

### 줄거리:
잠든 듯한 삶을 사는 한 남자와 교묘한 비누 판매자가 본능적인 남성성을 새로운 형태의 치료로 채널링하는 이야기를 다룹니다. 이들의 개념은 각 도시에서 언더그라운드 "격투 클럽"으로 번져 나가지만, 한 괴짜가 끼어들면서 혼돈의 나락으로 빠져들게 됩니다.

### 제작사:
- Fox 2000 Pictures
- Regency Enterprises
- Linson Entertainment
- 20th Century Fox
- Taurus Film

### 포스터:
![Fight Club Poster](https://image.tmdb.org/t/p/w780/jSziioSwPVrOy9Yow3XhWIBDjq1.jpg)

### 더 알아보기:
[Fight Club 홈페이지](https://www.20thcenturystudios.com/movies/fight-club)

궁금한 점이 있다면 말씀해 주세요!
